In [2]:
import torch
from chroma import Chroma, Protein, conditioners, api
device = 'cuda' if torch.cuda.is_available() else 'cpu'
#api.register_key(input("Enter API key: "))

In [4]:

api.register_key("ddbd677c928749d6b725a97cc5b56c65")

In [5]:
# Initialize the Model
chroma = Chroma()

# Sample a Protein
protein = chroma.sample()

Data saved to /var/folders/vy/ftzdqv0d5yndv7y8fr69mmdw0000gn/T/chroma_weights/90e339502ae6b372797414167ce5a632/weights.pt
Computing reference stats for 2g3n
Data saved to /var/folders/vy/ftzdqv0d5yndv7y8fr69mmdw0000gn/T/chroma_weights/03a3a9af343ae74998768a2711c8b7ce/weights.pt
Loaded from cache


Integrating SDE:   0%|          | 0/500 [00:00<?, ?it/s]

Potts Sampling:   0%|          | 0/500 [00:00<?, ?it/s]

Sequential decoding:   0%|          | 0/100 [00:00<?, ?it/s]

In [6]:
print(protein) # Inspect the sequence of the protein sample
protein.to('chroma_sample.cif') # Save the sample to disk
protein = Protein('chroma_sample.cif') # Load a protein from disk

Protein: system
> Chain A (100 residues)
DEEEKEREEKLRKALIELGFPADQPVTEDLIEEAKERILKKLEEMEKNLDQVSYEDAVRFANLALELEIDKERAEKFLKQVKEWYEKKQSLPEDYKPFEP




In [7]:
display(protein)

NGLWidget()

In [8]:
# Calculate sample scores
elbo = chroma.score(protein)['elbo'].score
print(f'sample elbo: {elbo}')

Integrating diffusion metrics:   0%|          | 0/50 [00:00<?, ?it/s]

sample elbo: 9.838735580444336


In [9]:
# Symmetry

In [10]:
SYMMETRY_GROUP = "C_3"
SUBUNIT_SIZES = [100]
KNBR = 2

In [11]:
# Draw a Sample
torch.manual_seed(68)
conditioner = conditioners.SymmetryConditioner(G=SYMMETRY_GROUP, num_chain_neighbors=KNBR)
symmetric_protein = chroma.sample(
    chain_lengths=SUBUNIT_SIZES,
    conditioner=conditioner,
    langevin_factor=8,
    inverse_temperature=8,
    sde_func="langevin",
    potts_symmetry_order=conditioner.potts_symmetry_order)

Integrating SDE:   0%|          | 0/500 [00:00<?, ?it/s]

Potts Sampling:   0%|          | 0/500 [00:00<?, ?it/s]

Sequential decoding:   0%|          | 0/300 [00:00<?, ?it/s]

In [12]:
symmetric_protein.to('symmetric_chroma_sample.cif')
display(symmetric_protein)

NGLWidget()

In [13]:
# Shape

In [15]:
LETTER = "G"
NUM_RESIDUES = 1000



In [ ]:
# Configure Shape Conditioner
from chroma.utility.chroma import letter_to_point_cloud
from PIL import ImageFont
ImageFont.truetype("/System/Library/Fonts/Supplemental/Arial.ttf", 64)

letter_point_cloud = letter_to_point_cloud(LETTER,font="/System/Library/Fonts/Supplemental/Arial.ttf")

conditioner = conditioners.ShapeConditioner(
        letter_point_cloud,
        chroma.backbone_network.noise_schedule,
        autoscale_num_residues=NUM_RESIDUES).to(device)



In [20]:
# Draw a Sample
torch.manual_seed(0)
shaped_protein = chroma.sample(chain_lengths=[NUM_RESIDUES], conditioner=conditioner)

Integrating SDE:   0%|          | 0/500 [00:00<?, ?it/s]

Potts Sampling:   0%|          | 0/500 [00:00<?, ?it/s]

Sequential decoding:   0%|          | 0/1000 [00:00<?, ?it/s]

In [21]:
shaped_protein.to('chroma_shape_sample.pdb')
display(shaped_protein)

NGLWidget()

In [22]:
type(letter_point_cloud)

numpy.ndarray

In [ ]:
letter_point_cloud.shape

(2000, 3)

: 